# LeetCode 1668 — Maximum Repeating Substring bằng Python

Link đề: https://leetcode.com/problems/maximum-repeating-substring/

Độ khó: **Easy** · Chủ đề: String, Dynamic Programming, String Matching

Skeleton LeetCode:

```python
class Solution:
    def maxRepeating(self, sequence: str, word: str) -> int:

```

Mục tiêu học:

1. Hiểu khái niệm **k-repeating**: `word` lặp `k` lần (`word * k`) là chuỗi con của `sequence`.
2. Nắm `str * k`, toán tử `in` với chuỗi, slicing `s[i:i+m]`.
3. Viết lời giải theo hint: thử `k = 1, 2, 3, ...` cho tới khi không còn là chuỗi con.
4. Nhận ra **tính đơn điệu** (k-repeating ⇒ (k-1)-repeating) để dùng **tìm kiếm nhị phân** trên `k`.
5. Viết lời giải **quy hoạch động** `dp[i]` = số lần lặp liên tiếp kết thúc tại `i`.
6. Thấy rõ bẫy "quét tham lam nhảy cóc" và `str.count` — hai lỗi kinh điển của bài này.

## 1. Bài toán yêu cầu gì?

Với chuỗi `sequence`, chuỗi `word` gọi là **k-repeating** nếu `word` nối với chính nó `k` lần là **chuỗi con** (substring — đoạn liên tiếp) của `sequence`.

**Giá trị k-repeating lớn nhất** của `word` là `k` lớn nhất thỏa mãn. Nếu `word` không phải chuỗi con của `sequence` thì là `0`.

| sequence | word | Kết quả | Vì sao |
|---|---|---:|---|
| `"ababc"` | `"ab"` | `2` | `"abab"` có trong `"ababc"` |
| `"ababc"` | `"ba"` | `1` | `"ba"` có, `"baba"` không |
| `"ababc"` | `"ac"` | `0` | `"ac"` không có |

Constraints:

- `1 <= sequence.length <= 100`
- `1 <= word.length <= 100`
- Chỉ chữ thường.

Chú ý: các bản lặp phải **liền nhau** (`"abab"`), không phải xuất hiện rời rạc (`"ab...ab"`).

In [1]:
sequence = "ababc"
for word in ["ab", "ba", "ac"]:
    print(f"word = {word!r}:")
    for k in range(1, 4):
        rep = word * k
        print(f"   k={k}: {rep!r:<10} có trong sequence? {rep in sequence}")

word = 'ab':
   k=1: 'ab'       có trong sequence? True
   k=2: 'abab'     có trong sequence? True
   k=3: 'ababab'   có trong sequence? False
word = 'ba':
   k=1: 'ba'       có trong sequence? True
   k=2: 'baba'     có trong sequence? False
   k=3: 'bababa'   có trong sequence? False
word = 'ac':
   k=1: 'ac'       có trong sequence? False
   k=2: 'acac'     có trong sequence? False
   k=3: 'acacac'   có trong sequence? False


## 2. Nền tảng Python cần biết

### 2.1. Nhân chuỗi `word * k`

`"ab" * 3 == "ababab"`. `"ab" * 0 == ""` (chuỗi rỗng).

In [2]:
print("ab" * 3)
print(repr("ab" * 0))
print(len("abc" * 4))

ababab
''
12


### 2.2. Toán tử `in` với chuỗi

`a in b` (với hai chuỗi) kiểm tra `a` có phải **chuỗi con liên tiếp** của `b` không. Chuỗi rỗng luôn là chuỗi con.

In [3]:
print("bab" in "ababc")
print("abc" in "ababc")
print("aac" in "ababc", "(ký tự có đủ nhưng không liên tiếp)")
print("" in "xyz")

True
True
False (ký tự có đủ nhưng không liên tiếp)
True


### 2.3. Slicing `s[i:i+m]` để so một đoạn

Lấy đoạn dài `m` bắt đầu tại `i`. Dùng để kiểm tra "tại vị trí `i` có `word` không".

In [4]:
s, w = "ababc", "ab"
m = len(w)
for i in range(len(s) - m + 1):
    print(f"i={i}: s[{i}:{i+m}] = {s[i:i+m]!r} == word? {s[i:i+m] == w}")

i=0: s[0:2] = 'ab' == word? True
i=1: s[1:3] = 'ba' == word? False
i=2: s[2:4] = 'ab' == word? True
i=3: s[3:5] = 'bc' == word? False


`range(len(s) - m + 1)`: vị trí bắt đầu cuối cùng hợp lệ là `len(s) - m`; cộng 1 vì `range` không gồm cận trên.

### 2.4. `str.find` và `str.startswith`

- `s.find(w)` trả về chỉ số xuất hiện **đầu tiên** của `w`, hoặc `-1`.
- `s.startswith(w, i)` kiểm tra `s` có `w` bắt đầu đúng tại `i` không (không cần tạo slice).

In [5]:
s = "ababc"
print(s.find("ba"), s.find("ac"))
print(s.startswith("ab", 0), s.startswith("ab", 2), s.startswith("ab", 1))

1 -1
True True False


## 3. Cách trực giác — thử k tăng dần (theo hint)

1. `k = 0`.
2. Trong khi `word * (k + 1)` còn là chuỗi con của `sequence` → `k += 1`.
3. Trả `k`.

Vì sao dừng ngay ở lần đầu tiên thất bại là đúng? Xem mục 4 (tính đơn điệu).

In [6]:
def max_repeating_bruteforce(sequence: str, word: str) -> int:
    k = 0
    while word * (k + 1) in sequence:
        k += 1
    return k

print(max_repeating_bruteforce("ababc", "ab"))
print(max_repeating_bruteforce("ababc", "ba"))
print(max_repeating_bruteforce("ababc", "ac"))

2
1
0


### Chi phí và nút thắt

Gọi `n = len(sequence)`, `m = len(word)`. `k` tối đa là `n // m`. Mỗi lần kiểm tra `in` trên chuỗi dài `(k+1)·m` tốn tới `O(n · (k+1)·m)` trong trường hợp xấu (cài đặt thực tế của CPython thường nhanh hơn nhiều).

Tổng xấu nhất cỡ `O(n · (n/m)² · m)`. Với `n <= 100` thì không sao, nhưng có hai câu hỏi đáng giá:

1. Có cần thử **từng** `k` một không? → **tìm kiếm nhị phân**.
2. Có thể **tận dụng kết quả** "đã có bao nhiêu bản lặp liền trước vị trí này" không? → **quy hoạch động**.

In [7]:
seq, w = "a" * 100, "a"
checks = 0
k = 0
while True:
    checks += 1
    if w * (k + 1) in seq:
        k += 1
    else:
        break
print("k =", k, "| số lần gọi `in` =", checks)

k = 100 | số lần gọi `in` = 101


## 4. Insight 1 — tính đơn điệu ⇒ tìm kiếm nhị phân

Nếu `word * k` là chuỗi con, thì `word * (k-1)` cũng vậy (nó là phần đầu của `word * k`). Vậy dãy đúng/sai theo `k` có dạng:

```text
k:      0    1    2    3    4    5
in?   True True True False False False
```

Một dãy "toàn True rồi toàn False" → tìm điểm chuyển bằng **binary search** trên `k ∈ [0, n // m]`.

In [8]:
sequence, word = "ababababc", "ab"
for k in range(0, len(sequence) // len(word) + 1):
    print(f"k={k}: {word * k in sequence}")

k=0: True
k=1: True
k=2: True
k=3: True
k=4: True


### Insight 2 — quy hoạch động theo vị trí kết thúc

Định nghĩa `dp[i]` = số bản `word` lặp **liên tiếp** mà khối đó **kết thúc đúng tại** vị trí `i` (tức `sequence[:i]` kết thúc bằng `word * dp[i]`).

Chuyển trạng thái (với `i >= m`):

```text
nếu sequence[i-m:i] == word:   dp[i] = dp[i-m] + 1
ngược lại:                      dp[i] = 0
```

Ý nghĩa: nếu `m` ký tự cuối là `word`, thì số bản lặp = 1 + số bản lặp kết thúc ngay trước đó (tại `i - m`).

Đáp án = `max(dp)`.

Tính tay `sequence = "ababc"`, `word = "ab"` (`m = 2`, `dp` dài `n + 1 = 6`):

| i | sequence[i-2:i] | == "ab"? | dp[i] |
|---:|---|:---:|---:|
| 0 | — | — | 0 |
| 1 | — | — | 0 |
| 2 | "ab" | ✔ | dp[0]+1 = 1 |
| 3 | "ba" | ✘ | 0 |
| 4 | "ab" | ✔ | dp[2]+1 = 2 |
| 5 | "bc" | ✘ | 0 |

`max(dp) = 2`.

In [9]:
sequence, word = "ababc", "ab"
m = len(word)
for i in range(m, len(sequence) + 1):
    print(f"i={i}: sequence[{i-m}:{i}] = {sequence[i-m:i]!r}")

i=2: sequence[0:2] = 'ab'
i=3: sequence[1:3] = 'ba'
i=4: sequence[2:4] = 'ab'
i=5: sequence[3:5] = 'bc'


## 5. Lời giải chính — quy hoạch động `O(n · m)`

Giảng từng đoạn:

**Đoạn 1 — chuẩn bị:** `n, m`, `dp = [0] * (n + 1)` (`dp[i]` dùng chỉ số "độ dài tiền tố" `0..n`).

**Đoạn 2 — duyệt `i` từ `m` đến `n`:** so `sequence[i-m:i]` với `word`.

**Đoạn 3 — cập nhật:** khớp → `dp[i] = dp[i-m] + 1`; không khớp giữ `0`.

**Đoạn 4 — trả `max(dp)`.**

In [10]:
def max_repeating_dp(sequence: str, word: str) -> int:
    n, m = len(sequence), len(word)
    if m > n:
        return 0
    dp = [0] * (n + 1)
    for i in range(m, n + 1):
        if sequence[i - m:i] == word:
            dp[i] = dp[i - m] + 1
    return max(dp)

print(max_repeating_dp("ababc", "ab"))
print(max_repeating_dp("ababc", "ba"))
print(max_repeating_dp("ababc", "ac"))
print(max_repeating_dp("ababaaba", "aba"))

2
1
0
2


## 6. TRACE từng bước

Trace một ví dụ "khó" — `sequence = "ababaaba"`, `word = "aba"` (đáp án `2`: `"abaaba"` bắt đầu tại chỉ số 2). Ví dụ này có hai lần xuất hiện **chồng lên nhau** (`"aba"` tại 0 và tại 2), đúng là chỗ mà các cách tham lam hay sai.

In [11]:
def trace_dp(sequence: str, word: str) -> int:
    n, m = len(sequence), len(word)
    dp = [0] * (n + 1)
    print(f"sequence={sequence!r}, word={word!r}, n={n}, m={m}")
    for i in range(m, n + 1):
        piece = sequence[i - m:i]
        if piece == word:
            dp[i] = dp[i - m] + 1
            print(f"i={i}: {piece!r} khớp  -> dp[{i}] = dp[{i-m}] + 1 = {dp[i]}")
        else:
            print(f"i={i}: {piece!r} không khớp -> dp[{i}] = 0")
    print("dp =", dp, "-> đáp án", max(dp))
    return max(dp)

trace_dp("ababaaba", "aba")

sequence='ababaaba', word='aba', n=8, m=3
i=3: 'aba' khớp  -> dp[3] = dp[0] + 1 = 1
i=4: 'bab' không khớp -> dp[4] = 0
i=5: 'aba' khớp  -> dp[5] = dp[2] + 1 = 1
i=6: 'baa' không khớp -> dp[6] = 0
i=7: 'aab' không khớp -> dp[7] = 0
i=8: 'aba' khớp  -> dp[8] = dp[5] + 1 = 2
dp = [0, 0, 0, 1, 0, 1, 0, 0, 2] -> đáp án 2


2

In [12]:
trace_dp("ababc", "ab")

sequence='ababc', word='ab', n=5, m=2
i=2: 'ab' khớp  -> dp[2] = dp[0] + 1 = 1
i=3: 'ba' không khớp -> dp[3] = 0
i=4: 'ab' khớp  -> dp[4] = dp[2] + 1 = 2
i=5: 'bc' không khớp -> dp[5] = 0
dp = [0, 0, 1, 0, 2, 0] -> đáp án 2


2

Đọc trace `"ababaaba"`: tại `i = 3` khối `"aba"` (chỉ số 0..2) cho `dp = 1`; tại `i = 5` khối `"aba"` (chỉ số 2..4) cho `dp[5] = dp[2] + 1 = 1` (vì `dp[2] = 0`); tại `i = 8` khối chỉ số 5..7 cho `dp[8] = dp[5] + 1 = 2`. DP **thử mọi điểm kết thúc**, nên không bỏ lỡ khối bắt đầu ở chỉ số 2.

## 7. Code nộp LeetCode

In [13]:
class Solution:
    def maxRepeating(self, sequence: str, word: str) -> int:
        n, m = len(sequence), len(word)
        dp = [0] * (n + 1)
        for i in range(m, n + 1):
            if sequence[i - m:i] == word:
                dp[i] = dp[i - m] + 1
        return max(dp)

sol = Solution()
assert sol.maxRepeating("ababc", "ab") == 2
assert sol.maxRepeating("ababc", "ba") == 1
assert sol.maxRepeating("ababc", "ac") == 0
print("3 ví dụ của đề PASS")

3 ví dụ của đề PASS


Khi `m > n`, `range(m, n + 1)` rỗng, `dp` toàn `0`, trả `0` — nên bản nộp không cần `if` riêng.

Bản ngắn theo hint (cũng được chấp nhận):

In [14]:
class SolutionSimple:
    def maxRepeating(self, sequence: str, word: str) -> int:
        k = 0
        while word * (k + 1) in sequence:
            k += 1
        return k

assert SolutionSimple().maxRepeating("ababc", "ab") == 2
print("SolutionSimple PASS")

SolutionSimple PASS


## 8. Các lời giải khác

### 8.1. Tìm kiếm nhị phân trên k

`lo = 0` (luôn đúng), `hi = n // m` (cận trên). Tìm `k` lớn nhất mà `word * k in sequence`.

Mẫu "tìm True cuối cùng": `mid = (lo + hi + 1) // 2` (làm tròn **lên** để tránh lặp vô hạn khi `lo = mid`).

In [15]:
def max_repeating_binary(sequence: str, word: str) -> int:
    lo, hi = 0, len(sequence) // len(word)
    while lo < hi:
        mid = (lo + hi + 1) // 2
        if word * mid in sequence:
            lo = mid            # mid đúng -> đáp án >= mid
        else:
            hi = mid - 1        # mid sai -> đáp án < mid
    return lo

print(max_repeating_binary("ababc", "ab"))
print(max_repeating_binary("ababaaba", "aba"))
print(max_repeating_binary("a" * 100, "a"))

2
2
100


In [16]:
def trace_binary(sequence, word):
    lo, hi = 0, len(sequence) // len(word)
    while lo < hi:
        mid = (lo + hi + 1) // 2
        ok = word * mid in sequence
        print(f"lo={lo}, hi={hi}, mid={mid}: {word!r}*{mid} in seq? {ok}")
        if ok:
            lo = mid
        else:
            hi = mid - 1
    print("đáp án:", lo)
    return lo

trace_binary("abababababxab", "ab")

lo=0, hi=6, mid=3: 'ab'*3 in seq? True
lo=3, hi=6, mid=5: 'ab'*5 in seq? True
lo=5, hi=6, mid=6: 'ab'*6 in seq? False
đáp án: 5


5

### 8.2. Thử mọi vị trí bắt đầu, đếm số bản lặp liên tiếp

Với mỗi `start`, đếm bao nhiêu bản `word` nối tiếp nhau bắt đầu tại `start` (dùng `startswith`). Lấy max. Đơn giản và đúng vì **thử mọi điểm bắt đầu**.

In [17]:
def max_repeating_starts(sequence: str, word: str) -> int:
    n, m = len(sequence), len(word)
    best = 0
    for start in range(n):
        cnt, pos = 0, start
        while sequence.startswith(word, pos):
            cnt += 1
            pos += m
        best = max(best, cnt)
    return best

print(max_repeating_starts("ababc", "ab"))
print(max_repeating_starts("ababaaba", "aba"))

2
2


### 8.3. Bảng so sánh các cách

Gọi `n = len(sequence)`, `m = len(word)`, `K = n // m`.

| Cách | Ý tưởng | Thời gian (xấu nhất, ước lượng) | Bộ nhớ | Nhận xét |
|---|---|---|---|---|
| Thử k tăng dần | `word*(k+1) in seq` | `O(K · n · K·m)` | `O(n)` | theo hint, ngắn nhất |
| Binary search k | tính đơn điệu | `O(log K · n · K·m)` | `O(n)` | ít lần gọi `in` hơn |
| Mọi vị trí bắt đầu | `startswith` liên tiếp | `O(n · K · m)` | `O(1)` | dễ hiểu, đúng |
| **DP kết thúc tại i** | `dp[i] = dp[i-m] + 1` | `O(n · m)` | `O(n)` | **nên dùng, ổn định** |

(Cận cho `in` là cận thô; thực tế CPython dùng thuật toán tìm chuỗi nhanh hơn.)

## 9. Vì sao DP đúng?

**Mệnh đề:** `dp[i]` = số `t` lớn nhất sao cho `sequence[:i]` kết thúc bằng `word * t`.

Chứng minh quy nạp theo `i`:

- `i < m`: không thể kết thúc bằng `word` (quá ngắn) → `dp[i] = 0`. Đúng.
- `i >= m`:
  - Nếu `sequence[i-m:i] != word` → không kết thúc bằng `word`, nên `t = 0`. Đúng.
  - Nếu bằng: `sequence[:i]` kết thúc bằng `word * t` (với `t >= 1`) ⇔ `sequence[:i-m]` kết thúc bằng `word * (t-1)`. Nên `t` lớn nhất = `dp[i-m] + 1`. Đúng.

Mọi khối `word * k` trong `sequence` đều **kết thúc** ở một vị trí `i` nào đó, nên `max(dp)` đúng bằng `k` lớn nhất.

**Vì sao vòng `while` của cách brute force dừng đúng chỗ:** do tính đơn điệu (mục 4), lần đầu tiên `word * (k+1)` không phải chuỗi con thì mọi `k' >= k + 1` cũng không, nên đáp án chính là `k`.

## 10. Độ phức tạp

Gọi `n = len(sequence)`, `m = len(word)`.

- **DP:** `n - m + 1` vòng, mỗi vòng so slice dài `m` → **Time `O(n · m)`**; **Space `O(n)`** cho `dp` (cộng `O(m)` tạm cho slice).
- **Thử k tăng dần:** tối đa `K + 1 = n/m + 1` lần gọi `in`; mỗi lần tạo chuỗi dài tới `n` và tìm trong `sequence`.
- **Binary search:** chỉ `O(log K)` lần gọi `in`.

Với `n, m <= 100`, mọi cách đều nhanh; DP có cận rõ ràng nhất.

## 11. Bộ test

1. 3 ví dụ đề.
2. `word` dài hơn `sequence` → `0`.
3. `word == sequence` → `1`.
4. Toàn một ký tự: `"aaaa"`, `"a"` → `4`; `"aaaa"`, `"aa"` → `2`.
5. Ca chồng lấp làm hỏng tham lam: `"ababaaba"`, `"aba"` → `2`.
6. Đối chiếu tất cả cách trên **mọi** chuỗi `a/b` ngắn (vét cạn) và ngẫu nhiên.

In [18]:
def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"{name:<26} PASS -> {actual}")

f = max_repeating_dp
check("đề 1", f("ababc", "ab"), 2)
check("đề 2", f("ababc", "ba"), 1)
check("đề 3", f("ababc", "ac"), 0)
check("word dài hơn", f("ab", "abc"), 0)
check("word == sequence", f("abc", "abc"), 1)
check("toàn 'a', word 'a'", f("aaaa", "a"), 4)
check("toàn 'a', word 'aa'", f("aaaaa", "aa"), 2)
check("chồng lấp 'aba'", f("ababaaba", "aba"), 2)
check("chồng lấp 'bab'", f("bababbab", "bab"), 2)

đề 1                       PASS -> 2
đề 2                       PASS -> 1
đề 3                       PASS -> 0
word dài hơn               PASS -> 0
word == sequence           PASS -> 1
toàn 'a', word 'a'         PASS -> 4
toàn 'a', word 'aa'        PASS -> 2
chồng lấp 'aba'            PASS -> 2
chồng lấp 'bab'            PASS -> 2


In [19]:
from itertools import product
import random

methods = [max_repeating_bruteforce, max_repeating_binary, max_repeating_starts,
           Solution().maxRepeating, SolutionSimple().maxRepeating]
count = 0
for n in range(1, 9):
    for seq in map("".join, product("ab", repeat=n)):
        for m in range(1, 4):
            for w in map("".join, product("ab", repeat=m)):
                ref = max_repeating_dp(seq, w)
                for meth in methods:
                    assert meth(seq, w) == ref, (getattr(meth, "__name__", meth), seq, w)
                count += 1
print("Vét cạn", count, "cặp (sequence, word) trên bảng chữ {a,b}: mọi cách khớp DP")

rng = random.Random(1668)
for _ in range(300):
    w = "".join(rng.choice("ab") for _ in range(rng.randint(1, 4)))
    seq = "".join(rng.choice([w, "a", "b"]) for _ in range(rng.randint(1, 25)))[:100]
    ref = max_repeating_dp(seq, w)
    assert all(meth(seq, w) == ref for meth in methods)
print("300 ca ngẫu nhiên (sequence chứa nhiều bản word) khớp")

Vét cạn 7140 cặp (sequence, word) trên bảng chữ {a,b}: mọi cách khớp DP
300 ca ngẫu nhiên (sequence chứa nhiều bản word) khớp


`product("ab", repeat=n)` sinh mọi tuple độ dài `n` gồm `a`/`b`; `map("".join, ...)` biến tuple thành chuỗi. Vét cạn trên miền nhỏ là cách rất mạnh để bắt lỗi logic.

## 12. Lỗi người mới hay gặp

### Lỗi 1: Dùng `sequence.count(word)`

`count` đếm số lần xuất hiện **không chồng lấp**, **không cần liền nhau**. Đề cần các bản **liền nhau**.

In [20]:
print("sai :", "abcab".count("ab"), "| đúng:", max_repeating_dp("abcab", "ab"))
print("sai :", "abcabcab".count("ab"), "| đúng:", max_repeating_dp("abcabcab", "ab"))

sai : 2 | đúng: 1
sai : 3 | đúng: 1


### Lỗi 2: Quét tham lam "nhảy cóc"

Ý tưởng: đi từ trái sang phải; khớp thì `cnt += 1; i += m`, không khớp thì `cnt = 0; i += 1`. Sai vì khi nhảy `m` bước, ta **bỏ qua** các điểm bắt đầu nằm giữa — và khối dài nhất có thể bắt đầu ở đó.

In [21]:
def wrong_greedy_jump(sequence, word):
    m, i, cnt, best = len(word), 0, 0, 0
    while i + m <= len(sequence):
        if sequence[i:i + m] == word:
            cnt += 1
            best = max(best, cnt)
            i += m
        else:
            cnt = 0
            i += 1
    return best

print("sai :", wrong_greedy_jump("ababaaba", "aba"))
print("đúng:", max_repeating_dp("ababaaba", "aba"))

sai : 1
đúng: 2


Giải thích: tham lam khớp `"aba"` tại 0, nhảy tới 3; tại 3 là `"baa"` không khớp, reset. Nó không bao giờ thử điểm bắt đầu 2, nơi có `"abaaba"`. DP và cách "mọi vị trí bắt đầu" không mắc lỗi này.

### Lỗi 3: Dùng `find` một lần rồi đếm từ đó

`sequence.find(word)` chỉ cho lần xuất hiện **đầu tiên**; khối dài nhất có thể nằm ở chỗ khác.

In [22]:
def wrong_find_once(sequence, word):
    start = sequence.find(word)
    if start == -1:
        return 0
    cnt = 0
    while sequence.startswith(word, start + cnt * len(word)):
        cnt += 1
    return cnt

print("sai :", wrong_find_once("abxababab", "ab"))
print("đúng:", max_repeating_dp("abxababab", "ab"))

sai : 1
đúng: 3


### Lỗi 4: Binary search làm tròn xuống → lặp vô hạn

Với mẫu "tìm True cuối cùng" (`lo = mid` khi đúng), nếu dùng `mid = (lo + hi) // 2` thì khi `hi = lo + 1` và `mid = lo` đúng, `lo` không đổi → kẹt mãi. Phải dùng `(lo + hi + 1) // 2`.

In [23]:
def wrong_binary(sequence, word, max_iter=20):
    lo, hi = 0, len(sequence) // len(word)
    it = 0
    while lo < hi and it < max_iter:
        mid = (lo + hi) // 2
        if word * mid in sequence:
            lo = mid
        else:
            hi = mid - 1
        it += 1
    return lo, hi, it

print("lo, hi, số vòng:", wrong_binary("ababc", "ab"), "(bị kẹt, phải chặn bằng max_iter)")

lo, hi, số vòng: (1, 2, 20) (bị kẹt, phải chặn bằng max_iter)


### Lỗi 5: Quên trường hợp `word` dài hơn `sequence`

Với DP, `range(m, n + 1)` tự rỗng nên an toàn. Nhưng nếu tự viết `dp[i - m]` với vòng bắt đầu từ `0` thì `i - m` âm — Python sẽ đọc **từ cuối list** (chỉ số âm) và cho kết quả sai mà không báo lỗi. Luôn bắt đầu từ `i = m`.

## 13. Mẹo nhận dạng pattern

- "Lặp **liên tiếp** bao nhiêu lần" → DP "**kết thúc tại i**": `dp[i] = dp[i - m] + 1`.
- Câu hỏi có/không theo `k` mà **đơn điệu** → **binary search trên đáp án**.
- Chuỗi có thể **chồng lấp** (overlap) → đừng tham lam nhảy cóc; thử mọi điểm bắt đầu/kết thúc.
- Constraints nhỏ + hint "brute force" → thử `k` tăng dần là đủ để AC.

Bài liên quan: 459 Repeated Substring Pattern, 686 Repeated String Match, 1566 Detect Pattern of Length M Repeated K or More Times, 28 Find the Index of the First Occurrence in a String (KMP).

## 14. Biến thể

### Biến thể 1: Trả về cả vị trí bắt đầu của khối dài nhất

In [24]:
def max_repeating_with_pos(sequence, word):
    n, m = len(sequence), len(word)
    dp = [0] * (n + 1)
    best, end = 0, 0
    for i in range(m, n + 1):
        if sequence[i - m:i] == word:
            dp[i] = dp[i - m] + 1
            if dp[i] > best:
                best, end = dp[i], i
    start = end - best * m if best else -1
    return best, start

print(max_repeating_with_pos("ababaaba", "aba"))
print(max_repeating_with_pos("ababc", "ac"))

(2, 2)
(0, -1)


### Biến thể 2: Đếm số vị trí kết thúc có ít nhất k bản lặp

In [25]:
def count_ends_at_least(sequence, word, k):
    n, m = len(sequence), len(word)
    dp = [0] * (n + 1)
    for i in range(m, n + 1):
        if sequence[i - m:i] == word:
            dp[i] = dp[i - m] + 1
    return sum(1 for v in dp if v >= k)

print(count_ends_at_least("abababab", "ab", 2))   # kết thúc tại 4, 6, 8

3


### Biến thể 3: Repeated String Match (LeetCode 686, rút gọn)

Số lần ít nhất phải lặp `a` để `b` là chuỗi con của `a * t` (hoặc `-1`). Cùng ý "thử `t` tăng dần", có cận trên rõ ràng.

In [26]:
def repeated_string_match(a, b):
    t = -(-len(b) // len(a))        # làm tròn lên len(b) / len(a)
    for extra in range(2):          # chỉ cần thử t và t + 1
        if b in a * (t + extra):
            return t + extra
    return -1

print(repeated_string_match("abcd", "cdabcdab"))
print(repeated_string_match("a", "aa"))
print(repeated_string_match("abc", "wxyz"))

3
2
-1


`-(-x // y)` là mẹo **chia làm tròn lên** với số nguyên dương: `-(-7 // 2) = -(-4) = 4`.

## 15. Bài tập tự luyện

Các cell TODO chỉ định nghĩa hàm, không được gọi khi chạy notebook.

### Bài 1 — Tính tay bảng dp

`sequence = "aaabaaaab"`, `word = "aa"`. Lập bảng `i, sequence[i-2:i], dp[i]` với `i = 0..9` và tìm đáp án.

### Bài 2 — Viết theo hint

In [27]:
def exercise_bruteforce(sequence: str, word: str) -> int:
    # TODO: while word * (k + 1) in sequence
    pass

# assert exercise_bruteforce("ababc", "ab") == 2

### Bài 3 — Viết lại DP

In [28]:
def exercise_dp(sequence: str, word: str) -> int:
    # TODO 1: dp = [0] * (n + 1)
    # TODO 2: for i in range(m, n + 1): so slice, cập nhật dp[i]
    # TODO 3: return max(dp)
    pass

# assert exercise_dp("ababaaba", "aba") == 2
# assert exercise_dp("ababc", "ac") == 0

### Bài 4 — Binary search trên k

In [29]:
def exercise_binary(sequence: str, word: str) -> int:
    # TODO: lo = 0, hi = n // m; mid làm tròn lên
    pass

# assert exercise_binary("abababababxab", "ab") == 5

### Bài 5 — Biến thể: số bản lặp liên tiếp dài nhất của **một ký tự**

Cho `s`, trả về độ dài đoạn dài nhất gồm toàn một ký tự giống nhau (LeetCode 1446 Consecutive Characters). Gợi ý: đây là DP với `m = 1` nhưng `word` thay đổi theo ký tự.

In [30]:
def exercise_max_power(s: str) -> int:
    # TODO: cur tăng khi s[i] == s[i-1], reset 1 khi khác
    pass

# assert exercise_max_power("leetcode") == 2
# assert exercise_max_power("abbcccddddeeeeedcba") == 5

### Bài 6 — Mini challenge: k-repeating lớn nhất cho **mọi** word trong danh sách

Cho `sequence` và list `words`. Trả về dict `{word: k}`. Không tạo `word * k` (chỉ dùng DP).

In [31]:
def exercise_all_words(sequence: str, words: list[str]) -> dict:
    # TODO: tái sử dụng hàm DP
    pass

# assert exercise_all_words("ababc", ["ab", "ba", "ac", "c"]) == {"ab": 2, "ba": 1, "ac": 0, "c": 1}

## 16. Gợi ý 3 mức

**Bài 1**
- Mức 1: `m = 2`, bắt đầu từ `i = 2`.
- Mức 2: `dp[2] = 1`, `dp[3] = dp[1] + 1 = 1`, `dp[4] = 0` ("ab").
- Mức 3: đoạn `"aaaa"` tại chỉ số 4..7 cho `dp[8] = dp[6] + 1 = 2`.

**Bài 2**
- Mức 1: `k = 0` trước vòng lặp.
- Mức 2: điều kiện `word * (k + 1) in sequence`.
- Mức 3: thân vòng chỉ là `k += 1`.

**Bài 3**
- Mức 1: `dp[i]` là số bản lặp kết thúc tại `i`.
- Mức 2: so `sequence[i - m:i] == word`.
- Mức 3: `dp[i] = dp[i - m] + 1`.

**Bài 4**
- Mức 1: `lo` luôn là giá trị đúng.
- Mức 2: `mid = (lo + hi + 1) // 2`.
- Mức 3: đúng → `lo = mid`; sai → `hi = mid - 1`.

**Bài 5**
- Mức 1: `cur = best = 1`.
- Mức 2: duyệt `i` từ 1.
- Mức 3: `cur = cur + 1 if s[i] == s[i-1] else 1`; `best = max(best, cur)`.

**Bài 6**
- Mức 1: dict comprehension.
- Mức 2: gọi DP cho từng word.
- Mức 3: `{w: dp_func(sequence, w) for w in words}`.

## 17. Đáp án tham khảo

**Bài 1:** `sequence = "aaabaaaab"` → `dp = [0,0,1,1,0,0,1,1,2,0]`, đáp án **2** (đoạn `"aaaa"` = `"aa" * 2`).

In [32]:
def answer_bruteforce(sequence, word):
    k = 0
    while word * (k + 1) in sequence:
        k += 1
    return k

def answer_dp(sequence, word):
    n, m = len(sequence), len(word)
    dp = [0] * (n + 1)
    for i in range(m, n + 1):
        if sequence[i - m:i] == word:
            dp[i] = dp[i - m] + 1
    return max(dp)

def answer_binary(sequence, word):
    lo, hi = 0, len(sequence) // len(word)
    while lo < hi:
        mid = (lo + hi + 1) // 2
        if word * mid in sequence:
            lo = mid
        else:
            hi = mid - 1
    return lo

def answer_max_power(s):
    cur = best = 1
    for i in range(1, len(s)):
        cur = cur + 1 if s[i] == s[i - 1] else 1
        best = max(best, cur)
    return best

def answer_all_words(sequence, words):
    return {w: answer_dp(sequence, w) for w in words}

def dp_table(sequence, word):
    n, m = len(sequence), len(word)
    dp = [0] * (n + 1)
    for i in range(m, n + 1):
        if sequence[i - m:i] == word:
            dp[i] = dp[i - m] + 1
    return dp

assert dp_table("aaabaaaab", "aa") == [0, 0, 1, 1, 0, 0, 1, 1, 2, 0]   # Bài 1
assert answer_dp("aaabaaaab", "aa") == 2
assert answer_bruteforce("ababc", "ab") == 2
assert answer_dp("ababaaba", "aba") == 2 and answer_dp("ababc", "ac") == 0
assert answer_binary("abababababxab", "ab") == 5
assert answer_max_power("leetcode") == 2
assert answer_max_power("abbcccddddeeeeedcba") == 5
assert answer_all_words("ababc", ["ab", "ba", "ac", "c"]) == {"ab": 2, "ba": 1, "ac": 0, "c": 1}
print("Tất cả đáp án PASS")

Tất cả đáp án PASS


## 18. Template nhớ nhanh + checklist submit

```python
class Solution:
    def maxRepeating(self, sequence: str, word: str) -> int:
        n, m = len(sequence), len(word)
        dp = [0] * (n + 1)                     # dp[i]: số bản lặp kết thúc tại i
        for i in range(m, n + 1):
            if sequence[i - m:i] == word:
                dp[i] = dp[i - m] + 1          # nối thêm 1 bản vào khối trước
        return max(dp)
```

Bản theo hint: `k = 0; while word * (k + 1) in sequence: k += 1; return k`.

Câu thần chú: **"Liên tiếp → kết thúc tại i, dp[i] = dp[i-m] + 1; đừng nhảy cóc."**

Checklist trước khi submit:

- [ ] Đếm bản lặp **liền nhau**, không dùng `str.count`?
- [ ] Không quét tham lam nhảy `m` bước (thử `"ababaaba"`, `"aba"` → `2`)?
- [ ] Vòng DP bắt đầu từ `i = m` (không có chỉ số âm)?
- [ ] `word` dài hơn `sequence` trả `0`?
- [ ] Nếu binary search: `mid` làm tròn **lên**?